#### Chroma
Chroma is a AI-native open-source vector database focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

Chroma provides an intuitive API for storing and querying embeddings alongside metadata and documents. It supports both in-memory and persistent storage, making it ideal for rapid prototyping and production applications alike. Chroma integrates seamlessly with LangChain for semantic search, retrieval-augmented generation (RAG), and other embedding-based workflows.

Key features of Chroma:
- simple, intuitive Python API for embedding storage and retrieval
- full-text search and metadata filtering capabilities
- flexible architecture supporting in-memory, disk, and client-server modes
- built-in support for various embedding models and distance metrics
- automatic handling of chunking, embedding, and storage management

Chroma is particularly well-suited for developers building AI applications that require fast prototyping, easy debugging, and transparent data handling. Whether building semantic search, chatbots, or RAG systems, Chroma simplifies the complexity of vector database management.

https://docs.langchain.com/oss/python/integrations/vectorstores/

In [7]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tqdm as notebook_tqdm

In [8]:
loader=TextLoader("speech.txt")
documents=loader.load()
documents

[Document(metadata={'source': 'speech.txt'}, page_content='Transformers are a type of neural network architecture that transforms or changes an input sequence into an output sequence. They do this by learning context and tracking relationships between sequence components. For example, consider this input sequence: "What is the color of the sky?" The transformer model uses an internal mathematical representation that identifies the relevancy and relationship between the words color, sky, and blue. It uses that knowledge to generate the output: "The sky is blue." \n\nOrganizations use transformer models for all types of sequence conversions, from speech recognition to machine translation and protein sequence analysis.\n\nWhy are transformers important?\nEarly deep learning models that focused extensively on natural language processing (NLP) tasks aimed at getting computers to understand and respond to natural human language. They guessed the next word in a sequence based on the previous 

In [9]:
#split
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
texts=text_splitter.split_documents(documents)
texts

[Document(metadata={'source': 'speech.txt'}, page_content='Transformers are a type of neural network architecture that transforms or changes an input sequence into an output sequence. They do this by learning context and tracking relationships between sequence components. For example, consider this input sequence: "What is the color of the sky?" The transformer model uses an internal mathematical representation that identifies the relevancy and relationship between the words color, sky, and blue. It uses that knowledge to generate the output: "The sky'),
 Document(metadata={'source': 'speech.txt'}, page_content='is blue."'),
 Document(metadata={'source': 'speech.txt'}, page_content='Organizations use transformer models for all types of sequence conversions, from speech recognition to machine translation and protein sequence analysis.\n\nWhy are transformers important?\nEarly deep learning models that focused extensively on natural language processing (NLP) tasks aimed at getting comput

In [10]:
embeddings=OllamaEmbeddings(model="gemma:2b")
db=Chroma.from_documents(texts, embeddings)
db

In [15]:
## Qyery db
query="How Organizations use transformer?"
docs=db.similarity_search(query)
docs[0].page_content


'Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'

In [14]:
## Saving to disk
vectordb=Chroma.from_documents(texts, embeddings, persist_directory="./chroma_db")

In [16]:
## loading from disk
db2=Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
docs=db2.similarity_search(query)
docs[0].page_content

'Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'

In [17]:
## Similarity serach with score
docs=db.similarity_search_with_score(query)
docs

[(Document(id='e7cb295e-a3e7-4449-90ae-7a857dc8e1a9', metadata={'source': 'speech.txt'}, page_content='Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'),
  0.32710522413253784),
 (Document(id='03db3353-f3de-4325-81a8-5027ee93a06c', metadata={'source': 'speech.txt'}, page_content='Organizations use transformer models for all types of sequence conversions, from speech recognition to machine translation and protein sequence analysis.\n\nWhy are transformers important?\nEarly deep learning models that focused extensively on natural language processing (NLP) tasks aimed at getting computers to understand and respond to natural human language. They guessed the next word in a sequence based on the previous word.'),
  0.4991731643676758),
 (Document(id='de3d9a59-cec1-452c-b834-58c7b196d956', metadata={'source': 'speech.txt'}, page_content='am from Italy. I like horse ri

In [19]:
### Retriever Options
retriever=vectordb.as_retriever()
retriever.invoke(query)[0].page_content


'Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'